# 📱 Task 3: SMS Spam Classification
## TF-IDF & Naive Bayes/SVM – CodSoft AI/ML Internship

**Google Colab Edition**: This notebook mounts Google Drive to save your models. It also includes an interactive **Web UI** at the end!

## 💾 Step 1: Mount Google Drive & Setup Directories

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/CodSoft_Task3'
MODELS_DIR = os.path.join(SAVE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'✅ Models will be saved to: {MODELS_DIR}')

## 🔑 Step 2: Download Dataset using Kaggle API

In [ ]:
import os
from getpass import getpass

print('🔑 Please enter your Kaggle API Token (starts with KGAT_...):')
token = getpass('Token: ')
os.environ['KAGGLE_API_TOKEN'] = token

DATA_DIR = '/content/dataset'
CSV_PATH = os.path.join(DATA_DIR, 'spam.csv')

if not os.path.exists(CSV_PATH):
    print('📥 Downloading dataset...')
    !pip install kaggle -q
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d uciml/sms-spam-collection-dataset -p {DATA_DIR} --unzip
    print('✅ Dataset downloaded!')
else:
    print('✅ Dataset already exists.')

## 📦 Step 3: Install Dependencies & Import Libraries

In [ ]:
!pip install pandas numpy scikit-learn nltk matplotlib seaborn wordcloud gradio joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from wordcloud import WordCloud
import gradio as gr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
print('✅ All libraries imported!')

## 📊 Step 4: Load & Explore the Dataset

In [ ]:
# The SMS dataset is typically latin-1 encoded
df = pd.read_csv(CSV_PATH, encoding='latin-1')

# Drop useless columns
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, errors='ignore')
# Rename for clarity
df.columns = ['Label', 'Message']

# Map labels to binary (ham=0, spam=1)
df['Label_Num'] = df['Label'].map({'ham': 0, 'spam': 1})

print(f"Dataset Shape: {df.shape}")
print(df.head())

# Plot Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Label', data=df, palette='Set2')
plt.title('Spam vs Ham Distribution')
plt.show()

print(f"\nClass counts:\n{df['Label'].value_counts()}")

## ☁️ Step 5: Word Cloud Visualization

In [ ]:
spam_messages = ' '.join(df[df['Label'] == 'spam']['Message'])
ham_messages = ' '.join(df[df['Label'] == 'ham']['Message'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Spam WordCloud
wc_spam = WordCloud(width=800, height=400, background_color='black', max_words=100).generate(spam_messages)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].set_title('Most Common SPAM Words', fontsize=16)
axes[0].axis('off')

# Ham WordCloud
wc_ham = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(ham_messages)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].set_title('Most Common HAM (Legit) Words', fontsize=16)
axes[1].axis('off')

plt.show()

## 🧹 Step 6: Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower() # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # remove punctuation/numbers
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

print("Applying text preprocessing...")
df['Clean_Message'] = df['Message'].apply(preprocess_text)
print("✅ Preprocessing complete!")

print("\nExample:")
print("Original:", df['Message'].iloc[2])
print("Cleaned :", df['Clean_Message'].iloc[2])

## 🔢 Step 7: Train/Test Split & Feature Extraction (TF-IDF)

In [ ]:
X = df['Clean_Message']
y = df['Label_Num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"X_train shape: {X_train_tfidf.shape}")
print(f"X_test shape: {X_test_tfidf.shape}")

## 🤖 Step 8: Train Models (Naive Bayes & SVM)

In [ ]:
print("Training Multinomial Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
print("✅ Naive Bayes Trained!")

print("\nTraining Support Vector Machine (SVM)...")
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_tfidf, y_train)
print("✅ SVM Trained!")

## 📈 Step 9: Evaluate Models

In [ ]:
def evaluate_model(model, name):
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n--- {name} Evaluation ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    plt.title(f'{name} - Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

evaluate_model(nb_model, "Naive Bayes")
evaluate_model(svm_model, "SVM")

## 💾 Step 10: Save Best Model (Naive Bayes)

In [ ]:
model_path = os.path.join(MODELS_DIR, 'spam_classifier_nb.pkl')
vectorizer_path = os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl')

joblib.dump(nb_model, model_path)
joblib.dump(tfidf, vectorizer_path)
print(f"✅ Model saved to: {model_path}")
print(f"✅ Vectorizer saved to: {vectorizer_path}")

## 🌐 Step 11: Interactive Web UI using Gradio (Bonus!)

In [ ]:
def predict_spam(message):
    # Preprocess and vectorize the input
    cleaned = preprocess_text(message)
    vectorized = tfidf.transform([cleaned])
    
    # Predict using Naive Bayes
    prediction = nb_model.predict(vectorized)[0]
    probability = nb_model.predict_proba(vectorized)[0].max()
    
    label = "🚨 SPAM" if prediction == 1 else "✅ HAM (Safe)"
    return f"{label} (Confidence: {probability*100:.2f}%)"

# Create the Gradio interface
interface = gr.Interface(
    fn=predict_spam,
    inputs=gr.Textbox(lines=4, placeholder="Enter SMS message here..."),
    outputs=gr.Textbox(label="Prediction Result"),
    title="📱 SMS Spam Detector",
    description="Enter a text message to classify it as Spam or Ham (Legitimate) using TF-IDF and Naive Bayes.",
    theme="default"
)

# Launch the UI in the notebook!
interface.launch(share=True)  # share=True creates a public link you can share!